# 第四课:Cosine Similarity——只看方向,不看长度

讲义:[04-cosine.md](04-cosine.md)。

## 1. 点积 vs cosine 对照实验(讲义 §3)

观察:长度差 10 倍的向量对,点积差 5~12 倍,cosine 纹丝不动。

In [ ]:
import numpy as np

pairs = [([2,0],[4,0]), ([1,1],[10,10]), ([3,0,1],[30,0,10]), ([1,0],[0,1]), ([1,0],[-1,0])]

print(f'{"dot":>8} {"cos":>9}   向量对')
for u, v in pairs:
    u, v = np.array(u, float), np.array(v, float)
    d = np.dot(u, v)
    c = d / (np.linalg.norm(u) * np.linalg.norm(v))
    print(f'{d:8.1f} {c:+9.4f}   {list(map(int,u))} vs {list(map(int,v))}')

## 2. 归一化之后,点积 = cosine(讲义 §4)

In [ ]:
def normalize(a):
    return a / np.linalg.norm(a)

a, b = np.array([1., 2, 3]), np.array([4., 5, 6])
cos_direct  = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
cos_via_dot = np.dot(normalize(a), normalize(b))

print('cosine             =', round(cos_direct, 4))
print('归一化后点积        =', round(cos_via_dot, 4))
print('相等:', np.isclose(cos_direct, cos_via_dot))
print('归一化后范数 =', round(np.linalg.norm(normalize(a)), 10))

## 3. 生产现场:cosine 检索 + 长度偏置反例(讲义 §5–6)

In [ ]:
def normalize_rows(L):
    return L / np.linalg.norm(L, axis=1, keepdims=True)   # axis=1: 每行(每条向量)自己除

def cosine_search(query, lib, top_k=3):
    q = normalize(query)
    L = normalize_rows(lib)
    scores = L @ q                      # 归一化后,点积即 cosine
    order = np.argsort(-scores)         # 降序
    return order[:top_k], scores[order[:top_k]]

# 反例构造:短文档方向正、长文档方向偏但范数巨大
query = np.array([1.0, 0.0, 0.0])
lib   = np.array([[1.0, 0.0, 0.0],      # 方向完全一致,但短
                  [30.0, 10.0, 0.0]])   # 方向偏了,但长 30 倍
print('裸点积分:', lib @ query, '→ 选', np.argmax(lib @ query))
idx, s = cosine_search(query, lib, top_k=2)
print('cosine 排序:', idx, '分数:', np.round(s, 4))

## 作业现场

- C3:关掉讲义重写 `cosine_search`(含 `np.argsort` 降序)。
- D2:浮点边界——把 `s` 打印到 12 位小数,看有没有超过 1 的值,试 `np.clip(s, -1, 1)`。
- E1:把第三课检索 demo 换成 cosine 版,确认长文档反例被修正。

In [ ]:
# TODO(C3): 独立重写 cosine_search,并自测
# 自测 1: query=[1,0,0], lib=[[1,0,0],[0,1,0]] → 第一条得分 1.0
# 自测 2: 上面反例中,选中的必须是「短而正」的那条

# TODO(D2): 观察浮点边界
q = normalize(np.array([1.0, 2.0, 3.0]))
print('cos(q, q) =', repr(float(q @ q)))   # 可能不是精确的 1.0
print('clip 后 =', float(np.clip(q @ q, -1, 1)))